In [1]:
import os
import numpy as np
import scipy as sp
import pandas as pd
from datetime import date
import marineHeatWaves as mhw
import netCDF4 as nc
import datetime
import matplotlib.pyplot as plt
from tqdm import notebook
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, Executor
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.backends.backend_agg import FigureCanvasAgg
import PIL.Image as Image
from scipy.stats import pearsonr
time=pd.date_range('2021-7-1','2021-9-30')

In [2]:
def getallname(s):
    sl=s.split()
    rs=''
    for i in sl:
        rs+=i
        rs+='_'
    return rs[:-1]

file=r"C:\Users\53150\OneDrive\heat_budget\data\ERA5slp2021.nc"
daset=nc.Dataset(file)

names=[]
for i in daset.variables.keys():
    print(f'{i}:{getallname(daset.variables[i].long_name)} ({daset.variables[i].units})')
    names.append(getallname(daset.variables[i].long_name))
    #print(np.array(daset.variables[i]).shape)

longitude:longitude (degrees_east)
latitude:latitude (degrees_north)
time:time (hours since 1900-01-01 00:00:00.0)
t2m:2_metre_temperature (K)
msl:Mean_sea_level_pressure (Pa)


In [3]:
msl=np.array(daset['msl'])
time=np.array(daset['time'])
lon=np.array(daset['longitude'])
lat=np.array(daset['latitude'])
Lon,Lat=np.meshgrid(lon,lat)
def cftime2pdtime(cf):
    return pd.to_datetime(datetime.datetime(cf.year,cf.month,cf.day,cf.hour))
time=pd.to_datetime(list(map(cftime2pdtime,nc.num2date(np.array(daset.variables['time']),daset.variables['time'].units))))

In [13]:
def fig2img(fig):
        """Convert a Matplotlib figure to a PIL Image and return it"""
        import io
        buf = io.BytesIO()
        fig.savefig(buf)
        buf.seek(0)
        img = Image.open(buf)
        return img
font_label = {'family': 'sans-serif',
                'size': 30}
font = {'family': 'sans-serif',
                'weight': 'normal'}
def draw(n):
    global msl,time,Lon,Lat
    import matplotlib 
    matplotlib.use('Agg')
    plt.figure(figsize=[10,4],dpi=300)
    ax = plt.axes(projection=ccrs.PlateCarree(central_longitude=0))
    plt.title(time[n].strftime('%y-%m-%d'))
    lon_formatter = LongitudeFormatter(zero_direction_label=False)
    lat_formatter = LatitudeFormatter()
    ax.xaxis.set_major_formatter(lon_formatter)
    ax.yaxis.set_major_formatter(lat_formatter)
    scale = '10m'
    land = cfeature.NaturalEarthFeature('physical', 'land', scale, edgecolor='face',facecolor=cfeature.COLORS['land'])
    ax.add_feature(land, facecolor='0.75')
    ax.coastlines()
    ax.set_xticks(range(-180, 1, 30))
    ax.set_yticks(range(0, 51, 10))
    plt.plot(np.arange(-180,-129,1),[35]*51,'r')
    plt.plot([-130]*16,np.arange(35,51,1),'r')
    c=plt.contourf(Lon,Lat,msl[n,:,:],np.arange(96100,103801,100),cmap='RdBu_r')
    plt.colorbar(c,orientation='horizontal',label='Mean sea level pressure (Pa)')
    image=fig2img(plt.gcf())
    return image

import warnings
warnings.filterwarnings("ignore")
images=list(map(draw,range(time.shape[0])))
im = images[0]
filename = 'test.gif'
im.save(fp=filename, format='gif', save_all=True, append_images=images[1:], duration=250,loop=0)
#draw(0)